In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from datetime import datetime

In [2]:
def parse_log_line(line):
    pattern = r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}).*?(\w+)\[(\d+)\]: (.+)'
    match = re.match(pattern, line)
    if match:
        return {
            'timestamp': match.group(1),
            'service': match.group(2),
            'pid': match.group(3),
            'message': match.group(4)
        }
    return None

# Load the log file
lines = []
with open('/var/log/auth.log', 'r') as f:
    for line in f:
        parsed = parse_log_line(line.strip())
        if parsed:
            lines.append(parsed)

df = pd.DataFrame(lines)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(df.shape)
print(df.head())

(187, 4)
            timestamp service  pid  \
0 2026-05-10 12:33:38  logind  202   
1 2026-05-10 12:33:43   login  391   
2 2026-05-10 12:33:43   login  391   
3 2026-05-10 12:33:44   login  391   
4 2026-05-10 12:33:44  logind  202   

                                             message  
0                                    New seat seat0.  
1  PAM unable to dlopen(pam_lastlog.so): /usr/lib...  
2           PAM adding faulty module: pam_lastlog.so  
3  pam_unix(login:session): session opened for us...  
4                        New session 1 of user root.  


In [3]:
print(df['service'].value_counts())

service
CRON        68
logind      46
login       38
usermod     20
polkitd      8
groupadd     3
useradd      1
passwd       1
chfn         1
gpasswd      1
Name: count, dtype: int64


In [4]:
# Filter for security-relevant events
security_keywords = ['failed', 'failure', 'invalid', 'error', 'sudo', 'root', 'session opened', 'session closed']

def flag_security(message):
    message_lower = message.lower()
    for keyword in security_keywords:
        if keyword in message_lower:
            return keyword
    return 'normal'

df['event_type'] = df['message'].apply(flag_security)
print(df['event_type'].value_counts())

event_type
normal            100
root               75
session opened     10
sudo                2
Name: count, dtype: int64


In [5]:
# Feature Engineering
df['hour'] = df['timestamp'].dt.hour
df['is_night'] = df['hour'].apply(lambda x: 1 if 2 <= x <= 5 else 0)
df['is_root'] = df['message'].apply(lambda x: 1 if 'root' in x.lower() else 0)
df['is_sudo'] = df['message'].apply(lambda x: 1 if 'sudo' in x.lower() else 0)
df['is_failed'] = df['message'].apply(lambda x: 1 if 'failed' in x.lower() or 'failure' in x.lower() else 0)
df['is_new_user'] = df['service'].apply(lambda x: 1 if x in ['useradd', 'usermod', 'groupadd'] else 0)

print(df[['timestamp', 'hour', 'is_night', 'is_root', 'is_sudo', 'is_failed', 'is_new_user']].head(10))
print("\nFeature summary:")
print(df[['is_night', 'is_root', 'is_sudo', 'is_failed', 'is_new_user']].sum())

            timestamp  hour  is_night  is_root  is_sudo  is_failed  \
0 2026-05-10 12:33:38    12         0        0        0          0   
1 2026-05-10 12:33:43    12         0        0        0          0   
2 2026-05-10 12:33:43    12         0        0        0          0   
3 2026-05-10 12:33:44    12         0        1        0          0   
4 2026-05-10 12:33:44    12         0        1        0          0   
5 2026-05-10 12:33:44    12         0        1        0          0   
6 2026-05-10 12:34:01    12         0        0        0          0   
7 2026-05-10 12:34:01    12         0        0        0          0   
8 2026-05-10 12:40:54    12         0        0        0          0   
9 2026-05-10 12:40:55    12         0        0        0          0   

   is_new_user  
0            0  
1            0  
2            0  
3            0  
4            0  
5            0  
6            0  
7            0  
8            0  
9            0  

Feature summary:
is_night       49
is_roo